# Shear-peak / filtered aperture-mass maps on HEALPix (DP2, Tract 9813)

Contact author: Alex Broughton  <br>Date: April 14, 2026


In [1]:
! eups list -s | grep lsst_distrib

lsst_distrib          gdfb3db0272+a02a4c3c89 	current v30_0_5 v30_0_5_rc1 setup


## Introduction

This notebook builds a **filtered aperture-mass map** (the shear-peak statistic used e.g. in DES-style analyses) for tract 9813 from DP2 metadetect `object_shear_all` via the Rubin Butler. It follows the same construction as `shear_peaks.ipynb`: optimal filter \(Q(\theta)\), tangential shear \(g_t\) about each cell center, aperture mass and noise-based \(\mathcal{S}/\mathcal{N}\), and a local-maximum peak definition. **Defaults are tuned toward galaxy clusters**: a larger aperture \(\theta_{\max}\) (smoothing / filter scale) than the small-field demo, and HEALPix cells small enough to sample that aperture with several pixels across its diameter. **Map cells are HEALPix pixels** (RING, `NSIDE` in `CONFIG`); each occupied pixel is evaluated at the pixel center (`hp.pix2ang`). Galaxies in the aperture are found with a `cKDTree` on RA/Dec (small-patch tangent plane). Shears use catalog `gauss_g1` / `gauss_g2`.

Approximate mean pixel spacing is \(\sim \sqrt{4\pi / (12\,N_{\mathrm{side}}^2)}\) rad on the sky; increasing `NSIDE` adds more cells and cost (`npix = 12 N_{\mathrm{side}}^2` if you ever materialize a full-sky map). This is **not** a Kaiser–Squires convergence map.


## 1.0 Set up

Imports, `CONFIG` (Butler, `NSIDE`, analysis knobs), and a Butler instance. The next code cell defines `Q(θ)`, tangential shear projection, aperture mass and `S/N`, and HEALPix neighbor-based peak finding. For **clusters**, co-tune `THETA_MAX_ARCMIN` (filter / smoothing scale, a few–15 arcmin is common) and `NSIDE` or `TARGET_CELL_ARCSEC` so the HEALPix linear scale is well below \(\theta_{\max}\) (several cells across the aperture).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import healpy as hp
from scipy.spatial import cKDTree
from tqdm.auto import tqdm

from lsst.daf.butler import Butler


def nside_from_mean_cell_arcsec(mean_arcsec):
    """Next power-of-two NSIDE with ~mean_arcsec mean pixel spacing (same scaling idea as shear_peaks)."""
    mean_arcsec = float(mean_arcsec)
    theta_rad = np.deg2rad(mean_arcsec / 3600.0)
    nside = int(np.ceil((4.0 * np.pi / 12.0) ** 0.5 / theta_rad))
    nside = max(1, nside)
    pow2 = 1 << int(np.ceil(np.log2(nside)))
    return int(pow2)


# --- Run configuration (edit these for a different tract / processing tag) ---
CONFIG = {
    # Rubin Butler datastore root (DP2 prep in this example).
    "REPO": "/sdf/data/rubin/repo/dp2_prep",
    # Collection that contains `object_shear_all` and skymap-linked datasets.
    "COLLECTION": "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage3",
    # Skymap dimension record attached to the Butler.
    "SKYMAP": "lsst_cells_v2",
    # Instrument dimension for LSSTCam DRP outputs.
    "INSTRUMENT": "LSSTCam",
    # Tract to load and to label figures.
    "TRACT": 9813,
    # Tangential aperture radius (arcmin); same scale enters Q(θ) and neighbor search.
    # Cluster-oriented: ~10–15' is typical for cluster-scale tangential shear in a single filter.
    "THETA_MAX_ARCMIN": 12.0,
    # Compactness in Q(θ); slightly larger than 0.15 weights the inner aperture a bit more (cluster cores).
    "X_C": 0.18,
    # HEALPix RING resolution (power of 2). If None, derived from TARGET_CELL_ARCSEC (~few arcmin cells).
    "NSIDE": 2048,
    "TARGET_CELL_ARCSEC": 110.0,
    # If True, union hp.query_disc around each occupied pixel (theta_max + margin) to fill holes.
    "HEALPIX_EXPAND_DISC": False,
    # Optional: require peaks to have S/N >= this value (set to None for no S/N cut).
    "PEAK_SN_MIN": 3.0,
}

_nside = CONFIG["NSIDE"]
if _nside is None:
    _nside = nside_from_mean_cell_arcsec(CONFIG["TARGET_CELL_ARCSEC"])
CONFIG["NSIDE"] = int(_nside)
# Approx. mean HEALPix spacing (arcsec) for this NSIDE (same sqrt(4π/12)/Nside scaling as shear_peaks).
_mean_arcsec = float(np.rad2deg((4.0 * np.pi / 12.0) ** 0.5 / CONFIG["NSIDE"]) * 3600.0)

%matplotlib inline
plt.rcParams.update({"font.size": 12})

butler = Butler(
    CONFIG["REPO"],
    collections=[CONFIG["COLLECTION"]],
    skymap=CONFIG["SKYMAP"],
    instrument=CONFIG["INSTRUMENT"],
)

_npix = hp.nside2npix(CONFIG["NSIDE"])
print("TRACT =", CONFIG["TRACT"])
print(
    "THETA_MAX_ARCMIN =",
    CONFIG["THETA_MAX_ARCMIN"],
    "| NSIDE =",
    CONFIG["NSIDE"],
    f"(npix={_npix}; ~mean spacing {_mean_arcsec:.1f} arcsec)",
)


In [ ]:
def Q(theta, theta_max, x_c):
    """Optimal filter Q(theta) from the shear-peaks / DES-style setup (see companion notebook).

    Parameters
    ----------
    theta, theta_max : array_like
        Angular separations in **degrees** (must share the same units). ``theta_max`` is the
        aperture radius in degrees (i.e. CONFIG['THETA_MAX_ARCMIN'] / 60).
    x_c : float
        Compactness parameter (default 0.15 in the example pipeline).
    """
    theta = np.asarray(theta, dtype=np.float64)
    x = theta / theta_max
    f = x / x_c
    with np.errstate(divide="ignore", invalid="ignore"):
        r = np.tanh(f) / f
    r *= 1.0 / (1.0 + np.exp(6.0 - 150.0 * theta) + np.exp(-47.0 + 50.0 * x))
    r = np.where(np.isfinite(r), r, 0.0)
    return r


def gamma_tx_sky(ra, dec, g1, g2, center):
    """Tangential (g_t) and cross (g_x) shear about ``center`` on the plane (deg, small-angle).

    Same role as ``lenspack.shear.gamma_tx`` in ``shear_peaks.ipynb``; implemented here so the
    stack kernel does not require ``lenspack``.
    """
    ra = np.asarray(ra, dtype=np.float64)
    dec = np.asarray(dec, dtype=np.float64)
    g1 = np.asarray(g1, dtype=np.float64)
    g2 = np.asarray(g2, dtype=np.float64)
    ra0, dec0 = center
    cos_dec0 = np.cos(np.deg2rad(dec0))
    dx = (ra - ra0) * cos_dec0
    dy = dec - dec0
    phi = np.arctan2(dy, dx)
    c2 = np.cos(2.0 * phi)
    s2 = np.sin(2.0 * phi)
    gt = -(g1 * c2 + g2 * s2)
    gx = -g1 * s2 + g2 * c2
    return gt, gx


def aperture_mass_and_sn_at_center(
    ra_c,
    dec_c,
    ra_all,
    dec_all,
    g1_all,
    g2_all,
    *,
    theta_max_arcmin,
    x_c,
    randomize=False,
):
    """Aperture mass (filtered) and S/N at one map pixel (``shear_peaks`` algorithm)."""
    theta_max_deg = float(theta_max_arcmin) / 60.0
    dra = ra_all - ra_c
    ddec = dec_all - dec_c
    dtheta = np.hypot(dra, ddec)
    m = dtheta <= theta_max_deg
    if not np.any(m):
        return np.nan, np.nan

    ra_s = ra_all[m]
    dec_s = dec_all[m]
    g1_s = g1_all[m]
    g2_s = g2_all[m]
    d_theta = dtheta[m]
    n_g = int(ra_s.size)

    gt, _gx = gamma_tx_sky(ra_s, dec_s, g1_s, g2_s, center=(ra_c, dec_c))

    if randomize:
        rng = np.random.default_rng()
        ang = rng.uniform(0.0, 2.0 * np.pi)
        gt = gt * (-np.cos(ang))

    Qi = Q(d_theta, theta_max=theta_max_deg, x_c=x_c)
    ap_mass = np.sum(Qi * gt) / n_g
    denom = np.sqrt(np.sum((Qi**2) * (g1_s**2 + g2_s**2)))
    if denom <= 0.0 or not np.isfinite(denom):
        return ap_mass, np.nan
    sn_ratio = np.sqrt(2.0) * (n_g * ap_mass) / denom
    return ap_mass, sn_ratio


def find_peaks_shear_healpix(
    ipix_eval,
    ap_mass,
    counts,
    sn_ratio,
    nside,
    theta_max_arcmin,
    sn_min=None,
):
    """Local maxima on aperture mass using RING pixel neighbors (``shear_peaks``-style cuts).

    A pixel is a peak if ``ap_mass`` is finite and strictly greater than ``ap_mass`` on every
    evaluated RING neighbor from ``hp.get_all_neighbours``. Same density cut as the 2D notebook:
    ``counts / (π θ_max²) > 0.5`` arcmin⁻² with ``θ_max`` in arcmin; optional ``sn_min`` on S/N.
    Neighbors not in ``ipix_eval`` are ignored for the local-maximum test.
    """
    ipix_eval = np.asarray(ipix_eval, dtype=np.int64)
    ap_mass = np.asarray(ap_mass, dtype=np.float64)
    counts = np.asarray(counts, dtype=np.float64)
    sn_ratio = np.asarray(sn_ratio, dtype=np.float64)
    idx_of = {int(ip): k for k, ip in enumerate(ipix_eval)}
    theta_max_arcmin = float(theta_max_arcmin)
    peaks_ipix = []

    for k, ip in enumerate(ipix_eval):
        ap = ap_mass[k]
        if not np.isfinite(ap):
            continue
        nbrs = hp.get_all_neighbours(nside, int(ip), nest=False)
        is_peak = True
        for nj in nbrs:
            if nj < 0:
                continue
            j = idx_of.get(int(nj))
            if j is None:
                continue
            if ap <= ap_mass[j]:
                is_peak = False
                break
        if not is_peak:
            continue
        density = counts[k] / (np.pi * (theta_max_arcmin**2))
        if density <= 0.5:
            continue
        if sn_min is not None:
            if not (np.isfinite(sn_ratio[k]) and sn_ratio[k] >= sn_min):
                continue
        peaks_ipix.append(int(ip))

    return np.asarray(peaks_ipix, dtype=np.int64)



## 2.0 Load catalogs

Read the tract-level `object_shear_all` dataset (all patches in one table for this tract). Memory use can be large; reduce footprint with column selection in Butler later if needed.


In [ ]:
data = butler.get("object_shear_all", dataId={"tract": CONFIG["TRACT"]})


## 3.0 Source selection

Keep metadetect **noshear** rows (`metaStep == "ns"`) and apply basic quality flags used in this demo.


In [ ]:
mask = (data.columns['metaStep'] == "ns")
mask &= (data['image_flags'] == 0)
mask &= (data['psfOriginal_flags'] == 0)
mask &= (data['bmask_flags'] == 0)
mask &= (data['ormask_flags'] == 0)
mask &= (data['mfrac'] < 0.1)

# Mag cut?

shear_catalog = data[mask]
print("Sources before quality cuts:", len(data))
print("Sources after quality cuts:", len(shear_catalog))


## 4.0 HEALPix cells and aperture-mass maps

Cells are **HEALPix RING** pixels at ``CONFIG['NSIDE']``. By default we evaluate **unique pixels** that contain at least one galaxy after selection (`hp.ang2pix` on RA/Dec). Set ``HEALPIX_EXPAND_DISC`` to True to union discs of radius ~``theta_max`` around each such pixel (more centers, slower). At each pixel center (`hp.pix2ang`) we use galaxies within ``THETA_MAX_ARCMIN`` with ``Q(θ)`` and ``g_t``. Parallel 1D arrays ``ipix_eval``, ``ap_mass_map``, ``sn_ratio_map``, ``counts_map`` store results (sparse over the sky).


In [ ]:
def _col(tbl, name):
    return tbl[name].combine_chunks().to_numpy(zero_copy_only=False).astype(np.float64, copy=False)

ra_deg = _col(shear_catalog, "ra")
dec_deg = _col(shear_catalog, "dec")
g1 = _col(shear_catalog, "gauss_g1")
g2 = _col(shear_catalog, "gauss_g2")

print("N galaxies:", ra_deg.size)


In [ ]:
theta_max_arcmin = CONFIG["THETA_MAX_ARCMIN"]
x_c = CONFIG["X_C"]
nside = int(CONFIG["NSIDE"])
theta_max_deg = theta_max_arcmin / 60.0

ipix_gal = hp.ang2pix(nside, ra_deg, dec_deg, nest=False, lonlat=True)
ipix_eval = np.unique(ipix_gal)

if CONFIG.get("HEALPIX_EXPAND_DISC", False):
    margin = 1.05
    radius_rad = np.deg2rad(theta_max_deg * margin)
    expanded = set(ipix_eval.tolist())
    for ip in tqdm(ipix_eval, desc="HEALPix expand (query_disc)"):
        vec = hp.pix2vec(nside, int(ip), nest=False)
        disc = hp.query_disc(nside, vec, radius_rad, inclusive=True, nest=False)
        expanded.update(int(x) for x in disc.tolist())
    ipix_eval = np.array(sorted(expanded), dtype=np.int64)

xy = np.column_stack([ra_deg, dec_deg])
tree = cKDTree(xy)

n_cell = int(ipix_eval.size)
ap_mass_map = np.full(n_cell, np.nan, dtype=np.float64)
sn_ratio_map = np.full(n_cell, np.nan, dtype=np.float64)
counts_map = np.zeros(n_cell, dtype=np.int32)

for k, ip in enumerate(tqdm(ipix_eval, desc="HEALPix cells")):
    ra_c, dec_c = hp.pix2ang(nside, int(ip), nest=False, lonlat=True)
    idx = tree.query_ball_point([float(ra_c), float(dec_c)], r=theta_max_deg)
    if not idx:
        continue
    idx = np.asarray(idx, dtype=np.int64)
    ra_s = ra_deg[idx]
    dec_s = dec_deg[idx]
    g1_s = g1[idx]
    g2_s = g2[idx]
    counts_map[k] = int(ra_s.size)
    ap_mass_map[k], sn_ratio_map[k] = aperture_mass_and_sn_at_center(
        float(ra_c),
        float(dec_c),
        ra_s,
        dec_s,
        g1_s,
        g2_s,
        theta_max_arcmin=theta_max_arcmin,
        x_c=x_c,
        randomize=False,
    )

print("N HEALPix cells evaluated:", n_cell)
print("Finite aperture-mass cells:", int(np.isfinite(ap_mass_map).sum()))


## 5.0 Peaks

Peaks are **strict local maxima** of aperture mass on the HEALPix RING graph (`hp.get_all_neighbours`), comparing only cells present in ``ipix_eval``. Same surface-density cut ``counts / (π θ_max²) > 0.5`` arcmin⁻² and optional ``PEAK_SN_MIN`` on ``sn_ratio_map`` as in ``shear_peaks.ipynb``.


In [ ]:
sn_min = CONFIG.get("PEAK_SN_MIN")
peak_ipix = find_peaks_shear_healpix(
    ipix_eval,
    ap_mass_map,
    counts_map,
    sn_ratio_map,
    nside,
    theta_max_arcmin=theta_max_arcmin,
    sn_min=sn_min,
)
print("N peaks (with current cuts):", int(peak_ipix.size))


## 6.0 Figures

Scatter maps in RA/Dec for evaluated HEALPix cells (aperture mass, ``S/N``, log counts). If ``12 NSIDE²`` is modest, an extra **gnomonic** HEALPix view of ``S/N`` is shown. Peaks are overlaid in celestial coordinates.


In [ ]:
ra_pix, dec_pix = hp.pix2ang(nside, ipix_eval, nest=False, lonlat=True)
ra_pix = np.asarray(ra_pix, dtype=np.float64)
dec_pix = np.asarray(dec_pix, dtype=np.float64)

msk = counts_map > 0
if msk.any():
    m_lo, m_hi = np.nanpercentile(ap_mass_map[msk], [5, 95])
    sn_lo, sn_hi = np.nanpercentile(sn_ratio_map[msk], [5, 95])
else:
    m_lo = m_hi = sn_lo = sn_hi = None

pt_size = max(1.0, min(8.0, 500_000.0 / max(int(ipix_eval.size), 1)))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), constrained_layout=True)
sc0 = axes[0].scatter(
    ra_pix[msk],
    dec_pix[msk],
    c=ap_mass_map[msk],
    s=pt_size,
    cmap="RdBu_r",
    vmin=m_lo,
    vmax=m_hi,
    linewidths=0,
    alpha=0.85,
)
axes[0].set_xlabel("RA (deg)")
axes[0].set_ylabel("Dec (deg)")
axes[0].set_title("Aperture mass (filtered)")
axes[0].invert_xaxis()
fig.colorbar(sc0, ax=axes[0], fraction=0.046, pad=0.02)

sc1 = axes[1].scatter(
    ra_pix[msk],
    dec_pix[msk],
    c=sn_ratio_map[msk],
    s=pt_size,
    cmap="RdBu_r",
    vmin=sn_lo,
    vmax=sn_hi,
    linewidths=0,
    alpha=0.85,
)
axes[1].set_xlabel("RA (deg)")
axes[1].set_ylabel("Dec (deg)")
axes[1].set_title(r"$\mathcal{S}/\mathcal{N}$")
axes[1].invert_xaxis()
fig.colorbar(sc1, ax=axes[1], fraction=0.046, pad=0.02)

logc = np.log10(counts_map.astype(np.float64) + 0.1)
sc2 = axes[2].scatter(
    ra_pix[msk],
    dec_pix[msk],
    c=logc[msk],
    s=pt_size,
    cmap="magma",
    linewidths=0,
    alpha=0.85,
)
axes[2].set_xlabel("RA (deg)")
axes[2].set_ylabel("Dec (deg)")
axes[2].set_title(r"$\log_{10}$(counts in $\theta_{\max}$) + 0.1")
axes[2].invert_xaxis()
fig.colorbar(sc2, ax=axes[2], fraction=0.046, pad=0.02)

fig.suptitle(
    f"Tract {CONFIG['TRACT']} — HEALPix NSIDE={nside} aperture mass / shear peaks "
    f"(theta_max={theta_max_arcmin} arcmin)",
    y=1.02,
)

fig2, axp = plt.subplots(figsize=(7, 5.5))
sc = axp.scatter(
    ra_pix[msk],
    dec_pix[msk],
    c=sn_ratio_map[msk],
    s=pt_size,
    cmap="RdBu_r",
    vmin=-4,
    vmax=4,
    linewidths=0,
    alpha=0.85,
)
if peak_ipix.size:
    ra_peak, dec_peak = hp.pix2ang(nside, peak_ipix, nest=False, lonlat=True)
    axp.scatter(
        ra_peak,
        dec_peak,
        s=120,
        facecolors="none",
        edgecolors="k",
        linewidths=1.2,
        label="peaks",
    )
axp.set_xlabel("RA (deg)")
axp.set_ylabel("Dec (deg)")
axp.set_title(r"$\mathcal{S}/\mathcal{N}$ with peak positions")
axp.invert_xaxis()
fig2.colorbar(sc, ax=axp, fraction=0.046, pad=0.02)
axp.legend(loc="upper right")
fig2.tight_layout()

# Dense gnomonic HEALPix view only when a full-sky map is small enough in memory.
MAX_DENSE_NPIX = 4_000_000
_npix_hp = hp.nside2npix(nside)
if _npix_hp <= MAX_DENSE_NPIX:
    rot_ra = float(np.median(ra_pix))
    rot_dec = float(np.median(dec_pix))
    sn_dense = np.full(_npix_hp, np.nan, dtype=np.float64)
    sn_dense[ipix_eval] = sn_ratio_map
    hp_sm = np.copy(sn_dense)
    hp_sm[~np.isfinite(hp_sm)] = hp.UNSEEN
    plt.figure(figsize=(7.0, 6.5))
    hp.gnomview(
        hp_sm,
        title=f"Tract {CONFIG['TRACT']} S/N (gnomonic, RING)",
        rot=(rot_ra, rot_dec),
        xsize=600,
        reso=0.35,
        cmap="RdBu_r",
        min=-4,
        max=4,
        nest=False,
    )
    if peak_ipix.size:
        ra_pk, dec_pk = hp.pix2ang(nside, peak_ipix, nest=False, lonlat=True)
        hp.projscatter(
            ra_pk,
            dec_pk,
            lonlat=True,
            marker="o",
            s=80,
            facecolors="none",
            edgecolors="k",
            linewidths=1.0,
        )
    plt.show()
